# Submitting Cirq Circuits to Azure Quantum with the QDK

This notebook demonstrates how to run Cirq `Circuit` jobs on Azure Quantum using `AzureQuantumService` from the QDK.

The workflow demonstrated here:

1. Build a Cirq `Circuit` with named measurement keys.
2. Reference an existing Azure Quantum workspace with `AzureQuantumService`.
3. Browse available targets via `service.targets()`.
4. Call `service.create_job(program=circuit, repetitions=..., target=...)` and fetch results (`job.results()`).

## Prerequisites

This notebook assumes the `qdk` package with Azure Quantum and Cirq support is installed. You can install everything with:

In [ ]:
%pip install "qdk[azure,cirq]"

This installs:
- The base `qdk` package (compiler, OpenQASM/QIR tooling)
- Azure Quantum client dependencies for submission
- Cirq for circuit construction

After installing, restart the notebook kernel if it was already running. You can verify installation with:

In [ ]:
import cirq, qdk, qdk.azure  # should import without errors

## Build a simple Cirq circuit

We start with a Bell-state circuit with two named measurement keys — one per qubit. Named keys are required so the result dictionary has clearly labeled registers.

In [ ]:
import cirq

q0, q1 = cirq.LineQubit.range(2)
circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, key="q0"),
    cirq.measure(q1, key="q1"),
)
print(circuit)

## Configure Azure Quantum workspace connection

To connect to an Azure workspace replace the following variables with your own values.

In [ ]:
subscription_id = 'xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx'
resource_group = 'myresourcegroup'
workspace_name = 'myworkspace'
location = 'westus'

## Submit the circuit to Azure Quantum

`AzureQuantumService` exposes Azure Quantum targets as Cirq-compatible target objects. When you call `service.targets()`, the SDK returns one of two kinds of target:

- **Provider-specific targets** — Some hardware vendors (e.g. IonQ, Quantinuum) ship dedicated Cirq target classes with native integration for their APIs, handling gate translation and result parsing using hardware-specific logic.
- **Generic QIR targets** — For any other target that accepts QIR input, the SDK automatically wraps it as an `AzureGenericQirCirqTarget`. These compile the Cirq circuit to OpenQASM 3 and then to QIR internally, so you don't have to manage those steps manually.

In practice `service.targets()` selects the right type for each target — you use the same `service.create_job()` call regardless of which target type is returned.

In [ ]:
from qdk.azure import Workspace
from azure.quantum.cirq import AzureQuantumService

workspace = Workspace(
    subscription_id=subscription_id,
    resource_group=resource_group,
    name=workspace_name,
    location=location,
)

service = AzureQuantumService(workspace)

# List available targets
for target in service.targets():
    print(f"{target.name:45s}  {type(target).__name__}")

In [ ]:
from collections import Counter

# Replace with any target name from the list above
target_name = "rigetti.sim.qvm"

job = service.create_job(
    program=circuit,
    repetitions=100,
    name="cirq-bell-job",
    target=target_name,
)
print(f"Job {job.job_id()} submitted — waiting for results...")

result = job.results()

# Combine separate measurement keys into joint bitstrings
keys = sorted(result.measurements.keys())
joint = Counter(
    "".join(str(int(result.measurements[k][i][0])) for k in keys)
    for i in range(len(result.measurements[keys[0]]))
)
total = sum(joint.values())
print(f"\nResults ({total} shots)  [keys: {', '.join(keys)}]:")
for bitstring, count in sorted(joint.items()):
    print(f"  {bitstring}: {count:4d}  ({count/total:.1%})")

## Handling qubit loss on noisy hardware

On some hardware backends — particularly neutral-atom and trapped-ion devices — a qubit may be lost before measurement (e.g. an atom is ejected from the trap). When this happens, the backend records `"-"` in the bitstring position for that qubit rather than `"0"` or `"1"`. Because loss shots contain non-binary characters, they cannot be included in standard measurement arrays, which assume a fixed binary alphabet. The SDK therefore separates them automatically.

The `cirq.ResultDict` returned by `job.results()` exposes two ways to access shots:

| Field | What it contains |
|---|---|
| **`result.measurements[key]`** | NumPy int8 array of accepted shots only (no `"-"`), shape `(accepted_shots, num_qubits)` |
| **`result.raw_measurements()[key]`** | String array of all shots (loss shots have `"-"`), same key structure as `measurements` |
| **`result.raw_shots`** | The original shot objects exactly as returned by the backend |

Use `result.measurements` for any downstream analysis that expects clean binary arrays. Use `result.raw_measurements()` and `result.raw_shots` to inspect loss patterns — for example, to calculate the overall loss rate or identify which qubit positions are being lost most frequently.

> **Tip**: A high loss rate may indicate hardware instability or a circuit that is too deep for the current calibration.